In [2]:
import spacy
import json
from spacy import displacy
from scrapy.selector import Selector
from spacy.pipeline.senter import DEFAULT_SENTER_MODEL

In [6]:
with open('../projscrape/posts.json') as fp:
    articles = json.load(fp)

nlp = spacy.load('en_core_web_sm')

with open('disease_list.json') as fp:
    DISEASES = [d['name'].lower() for d in json.load(fp)]
    
with open("syndrome_list.json") as fp:
    SYNDROMES = [d['name'].lower() for d in json.load(fp)]

# add a pipeline to detect syndromes
ruler = nlp.add_pipe("entity_ruler", config={
    "phrase_matcher_attr": "LOWER",
})
# print([{"label": "DISEASE", "pattern": d} for d in nlp.pipe(DISEASES)])
ruler.add_patterns([{"label": "DISEASE", "pattern": d} for d in DISEASES])
ruler.add_patterns([{"label": "SYNDROME", "pattern": s} for s in SYNDROMES])

In [7]:
def parse_text(article):
    html = article['article_text']
    body = Selector(text=html)
    text = ' '.join(s.strip() for s in body.css('#content *::text').getall())
    # break into paragraphs
    for item in body.css('p'):
        yield ' '.join(item.css("*::text").getall())


In [10]:
def parse_reports(paragraphs):
    docs = nlp.pipe(paragraphs)
    for doc in docs:
        with_ent = lambda x: (ent for ent in doc.ents if ent.label_ == x)

        diseases = with_ent("DISEASE")
        syndromes = with_ent("SYNDROME")
        dates = with_ent("DATE")
        locations = with_ent("GPE") # countries, cities and states

        if (any(diseases) or any(syndromes)) and any(dates) and any(locations):
            spacy.displacy.render(doc, style="dep")
    
for i in range(4):
    parse_reports(parse_text(articles[i]))
    

